# KoELECTRA Baseline

Training and evaluation of the second model-family baseline.

Set the private project root before execution:

```python
import os
os.environ['KLJPAUDIT_ROOT'] = '/path/to/K-LJPAUDIT'
```

Do not publish raw judicial facts, transformed text, per-case predictions, case identifiers, review files, or model checkpoints.

## Execution block 1

In [ ]:
import os
# ============================================================
# K-LJPAUDIT
# SECOND MODEL FAMILY GENERALIZATION — KoELECTRA BASELINE
# Seed 42, matched training protocol
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import gc
import json
import math
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(os.environ.get("KLJPAUDIT_ROOT", "."))

EXPERIMENT_NAME = "koelectra_baseline_generalization_v1"

RESULT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / EXPERIMENT_NAME
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "koelectra_base_v3_multitask_seed42"
)

PREDICTION_DIR = RESULT_ROOT / "predictions"
METRICS_DIR = RESULT_ROOT / "metrics"

for path in [
    RESULT_ROOT,
    CHECKPOINT_DIR,
    PREDICTION_DIR,
    METRICS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
DATASET_NAME = "lbox/lbox_open"
DATASET_CONFIG = "ljp_criminal"

SEED = 42

TARGETS = [
    "fine_lv",
    "imprisonment_with_labor_lv",
    "imprisonment_without_labor_lv",
]

NUM_LABELS = {
    "fine_lv": 5,
    "imprisonment_with_labor_lv": 6,
    "imprisonment_without_labor_lv": 5,
}

MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
DROPOUT = 0.10
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
USE_AMP = DEVICE.type == "cuda"

print("=" * 72)
print("SECOND MODEL FAMILY — KoELECTRA BASELINE")
print("=" * 72)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print("PyTorch:", torch.__version__)
print("Model:", MODEL_NAME)
print("Result root:", RESULT_ROOT)
print("Checkpoint directory:", CHECKPOINT_DIR)


# ------------------------------------------------------------
# 2. Reproducibility
# ------------------------------------------------------------

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass


set_all_seeds(SEED)


# ------------------------------------------------------------
# 3. Load dataset
# ------------------------------------------------------------

print("\nLoading LBox criminal LJP dataset...")

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
)

if "valid" in dataset and "validation" not in dataset:
    dataset["validation"] = dataset["valid"]

required_splits = {
    "train",
    "validation",
    "test",
}

missing_splits = required_splits - set(dataset.keys())

assert not missing_splits, (
    f"Missing splits: {sorted(missing_splits)}; "
    f"available={list(dataset.keys())}"
)

train_raw = dataset["train"]
valid_raw = dataset["validation"]
test_raw = dataset["test"]

assert len(train_raw) == 8400
assert len(valid_raw) == 1050
assert len(test_raw) == 1050

print("Train:", len(train_raw))
print("Validation:", len(valid_raw))
print("Test:", len(test_raw))


def extract_labels(example):
    label = example["label"]

    return {
        target: int(label[target])
        for target in TARGETS
    }


for split_name, split_data in [
    ("train", train_raw),
    ("validation", valid_raw),
    ("test", test_raw),
]:
    assert "id" in split_data.column_names
    assert "facts" in split_data.column_names
    assert "label" in split_data.column_names

    for index in range(len(split_data)):
        row = split_data[index]
        text = str(row["facts"]).strip()

        assert text, (
            f"{split_name}[{index}] contains empty facts"
        )

        labels = extract_labels(row)

        for target in TARGETS:
            assert 0 <= labels[target] < NUM_LABELS[target], (
                f"{split_name}[{index}] invalid "
                f"{target}={labels[target]}"
            )

print("Dataset validation: PASSED")


# ------------------------------------------------------------
# 4. Tokenizer and Dataset
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=False,
)

print("Tokenizer:", tokenizer.__class__.__name__)
print(
    "Tokenizer model maximum:",
    tokenizer.model_max_length,
)


class LJPDataset(Dataset):
    def __init__(self, split):
        self.split = split

    def __len__(self):
        return len(self.split)

    def __getitem__(self, index):
        row = self.split[index]
        labels = extract_labels(row)

        encoded = tokenizer(
            str(row["facts"]),
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_attention_mask=True,
        )

        output = {
            "case_uid": str(row["id"]),
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
        }

        if "token_type_ids" in encoded:
            output["token_type_ids"] = (
                encoded["token_type_ids"]
            )

        for target in TARGETS:
            output[target] = labels[target]

        return output


def collate_batch(batch):
    token_features = []

    for row in batch:
        feature = {
            "input_ids": row["input_ids"],
            "attention_mask": row["attention_mask"],
        }

        if "token_type_ids" in row:
            feature["token_type_ids"] = (
                row["token_type_ids"]
            )

        token_features.append(feature)

    padded = tokenizer.pad(
        token_features,
        padding=True,
        return_tensors="pt",
    )

    output = {
        "case_uid": [
            row["case_uid"]
            for row in batch
        ],
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
    }

    if "token_type_ids" in padded:
        output["token_type_ids"] = (
            padded["token_type_ids"]
        )

    for target in TARGETS:
        output[target] = torch.tensor(
            [row[target] for row in batch],
            dtype=torch.long,
        )

    return output


train_dataset = LJPDataset(train_raw)
valid_dataset = LJPDataset(valid_raw)
test_dataset = LJPDataset(test_raw)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    collate_fn=collate_batch,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    collate_fn=collate_batch,
)


# ------------------------------------------------------------
# 5. Sqrt-inverse class weights
# ------------------------------------------------------------

class_weights = {}
class_distribution = {}

for target in TARGETS:
    values = np.asarray(
        [
            int(row["label"][target])
            for row in train_raw
        ],
        dtype=np.int64,
    )

    counts = np.bincount(
        values,
        minlength=NUM_LABELS[target],
    ).astype(np.float64)

    assert np.all(counts > 0), (
        f"{target}: zero-support class found"
    )

    weights = 1.0 / np.sqrt(counts)
    weights = weights / weights.mean()

    class_weights[target] = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    class_distribution[target] = {
        "counts": counts.astype(int).tolist(),
        "weights": weights.tolist(),
    }

    print(f"\n[{target}]")
    print(" Counts:", counts.astype(int).tolist())
    print(
        " Weights:",
        np.round(weights, 6).tolist(),
    )


# ------------------------------------------------------------
# 6. KoELECTRA multi-task classifier
# ------------------------------------------------------------

class KoElectraMultiTaskClassifier(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels,
        dropout,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=False,
        )

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.classifiers = nn.ModuleDict({
            target: nn.Linear(
                hidden_size,
                target_num_labels,
            )
            for target, target_num_labels
            in num_labels.items()
        })

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
    ):
        model_inputs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        if token_type_ids is not None:
            model_inputs["token_type_ids"] = (
                token_type_ids
            )

        outputs = self.encoder(**model_inputs)

        cls_representation = (
            outputs.last_hidden_state[:, 0, :]
        )

        cls_representation = self.dropout(
            cls_representation
        )

        return {
            target: classifier(cls_representation)
            for target, classifier
            in self.classifiers.items()
        }


model = KoElectraMultiTaskClassifier(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

loss_functions = {
    target: nn.CrossEntropyLoss(
        weight=class_weights[target]
    )
    for target in TARGETS
}

print("\nEncoder architecture:")
print(" ", model.encoder.__class__.__name__)
print(" Hidden size:", model.encoder.config.hidden_size)
print(
    " Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)


# ------------------------------------------------------------
# 7. Evaluation
# ------------------------------------------------------------

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    case_uids = []
    true_values = {
        target: []
        for target in TARGETS
    }
    predicted_values = {
        target: []
        for target in TARGETS
    }

    for batch in loader:
        model_inputs = {
            "input_ids": batch["input_ids"].to(
                DEVICE,
                non_blocking=True,
            ),
            "attention_mask": batch[
                "attention_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            ),
        }

        if "token_type_ids" in batch:
            model_inputs["token_type_ids"] = (
                batch["token_type_ids"].to(
                    DEVICE,
                    non_blocking=True,
                )
            )

        logits = model(**model_inputs)

        case_uids.extend(batch["case_uid"])

        for target in TARGETS:
            true_values[target].extend(
                batch[target].cpu().numpy().tolist()
            )

            predicted_values[target].extend(
                logits[target]
                .argmax(dim=-1)
                .cpu()
                .numpy()
                .tolist()
            )

    metrics = {}
    target_macro_f1_values = []

    for target in TARGETS:
        labels = list(range(NUM_LABELS[target]))

        accuracy = accuracy_score(
            true_values[target],
            predicted_values[target],
        )

        macro_f1 = f1_score(
            true_values[target],
            predicted_values[target],
            labels=labels,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            true_values[target],
            predicted_values[target],
            labels=labels,
            average="weighted",
            zero_division=0,
        )

        metrics[target] = {
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
        }

        target_macro_f1_values.append(macro_f1)

    true_matrix = np.column_stack([
        np.asarray(true_values[target])
        for target in TARGETS
    ])

    prediction_matrix = np.column_stack([
        np.asarray(predicted_values[target])
        for target in TARGETS
    ])

    metrics["aggregate"] = {
        "mean_macro_f1": float(
            np.mean(target_macro_f1_values)
        ),
        "joint_exact_match": float(
            np.mean(
                np.all(
                    true_matrix == prediction_matrix,
                    axis=1,
                )
            )
        ),
    }

    prediction_df = pd.DataFrame({
        "case_uid": case_uids,
    })

    for target in TARGETS:
        prediction_df[f"{target}_true"] = (
            true_values[target]
        )
        prediction_df[f"pred_{target}"] = (
            predicted_values[target]
        )

    return metrics, prediction_df


# ------------------------------------------------------------
# 8. Optimizer and scheduler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)

total_training_steps = (
    updates_per_epoch * NUM_EPOCHS
)

warmup_steps = int(
    total_training_steps * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("\nTraining configuration:")
print(" Seed:", SEED)
print(" Epochs:", NUM_EPOCHS)
print(" Learning rate:", LEARNING_RATE)
print(" Train batch size:", TRAIN_BATCH_SIZE)
print(" Gradient accumulation:", GRAD_ACCUM_STEPS)
print(
    " Effective batch size:",
    TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
)
print("Updates per epoch:", updates_per_epoch)
print("Total updates:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("FP16:", USE_AMP)


# ------------------------------------------------------------
# 9. Train
# ------------------------------------------------------------

checkpoint_path = CHECKPOINT_DIR / "best_model.pt"

best_valid_mean_macro_f1 = -np.inf
best_epoch = None
training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_update_count = 0

    for step, batch in enumerate(
        train_loader,
        start=1,
    ):
        model_inputs = {
            "input_ids": batch["input_ids"].to(
                DEVICE,
                non_blocking=True,
            ),
            "attention_mask": batch[
                "attention_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            ),
        }

        if "token_type_ids" in batch:
            model_inputs["token_type_ids"] = (
                batch["token_type_ids"].to(
                    DEVICE,
                    non_blocking=True,
                )
            )

        labels = {
            target: batch[target].to(
                DEVICE,
                non_blocking=True,
            )
            for target in TARGETS
        }

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = model(**model_inputs)

            target_losses = [
                loss_functions[target](
                    logits[target],
                    labels[target],
                )
                for target in TARGETS
            ]

            loss = torch.stack(
                target_losses
            ).mean()

            backward_loss = (
                loss / GRAD_ACCUM_STEPS
            )

        scaler.scale(backward_loss).backward()

        running_loss += float(
            loss.detach().cpu()
        )

        should_update = (
            step % GRAD_ACCUM_STEPS == 0
            or step == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            optimizer_update_count += 1

        if step % 200 == 0 or step == len(train_loader):
            print(
                f"Epoch {epoch}/{NUM_EPOCHS} | "
                f"batch {step}/{len(train_loader)} | "
                f"loss={running_loss / step:.6f}"
            )

    valid_metrics, _ = evaluate(
        model,
        valid_loader,
    )

    valid_mean_macro_f1 = (
        valid_metrics["aggregate"][
            "mean_macro_f1"
        ]
    )

    valid_joint_exact_match = (
        valid_metrics["aggregate"][
            "joint_exact_match"
        ]
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": (
            running_loss / len(train_loader)
        ),
        "optimizer_updates": (
            optimizer_update_count
        ),
        "valid_mean_macro_f1": (
            valid_mean_macro_f1
        ),
        "valid_joint_exact_match": (
            valid_joint_exact_match
        ),
        "valid_target_metrics": {
            target: valid_metrics[target]
            for target in TARGETS
        },
    }

    training_history.append(epoch_record)

    print(
        f"\nEpoch {epoch} validation | "
        f"mean Macro-F1="
        f"{valid_mean_macro_f1:.6f} | "
        f"joint EM="
        f"{valid_joint_exact_match:.6f}"
    )

    if (
        valid_mean_macro_f1
        > best_valid_mean_macro_f1
    ):
        best_valid_mean_macro_f1 = (
            valid_mean_macro_f1
        )
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": (
                    model.state_dict()
                ),
                "seed": SEED,
                "epoch": epoch,
                "model_name": MODEL_NAME,
                "architecture": (
                    "KoElectraMultiTaskClassifier"
                ),
                "num_labels": NUM_LABELS,
                "max_length": MAX_LENGTH,
                "valid_mean_macro_f1": (
                    valid_mean_macro_f1
                ),
                "valid_joint_exact_match": (
                    valid_joint_exact_match
                ),
                "training_protocol": {
                    "epochs": NUM_EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "train_batch_size": (
                        TRAIN_BATCH_SIZE
                    ),
                    "eval_batch_size": (
                        EVAL_BATCH_SIZE
                    ),
                    "gradient_accumulation_steps": (
                        GRAD_ACCUM_STEPS
                    ),
                    "weight_decay": WEIGHT_DECAY,
                    "warmup_ratio": WARMUP_RATIO,
                    "dropout": DROPOUT,
                    "class_weighting": (
                        "sqrt_inverse_mean_normalized"
                    ),
                },
            },
            checkpoint_path,
        )

        print(
            "Best checkpoint updated:",
            checkpoint_path,
        )


# ------------------------------------------------------------
# 10. Reload best checkpoint and evaluate test set
# ------------------------------------------------------------

assert checkpoint_path.exists()

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

assert not load_result.missing_keys
assert not load_result.unexpected_keys

print("\nCheckpoint loading:")
print(" Missing keys:", load_result.missing_keys)
print(
    " Unexpected keys:",
    load_result.unexpected_keys,
)

test_metrics, predictions = evaluate(
    model,
    test_loader,
)

assert len(predictions) == 1050
assert predictions["case_uid"].notna().all()

predictions["case_uid"] = (
    predictions["case_uid"].astype(str)
)

assert predictions["case_uid"].nunique() == 1050


# ------------------------------------------------------------
# 11. Save private predictions and public metrics
# ------------------------------------------------------------

private_prediction_path = (
    PREDICTION_DIR
    / "koelectra_seed42_test_predictions_private.parquet"
)

predictions.to_parquet(
    private_prediction_path,
    index=False,
)

target_metric_rows = []

for target in TARGETS:
    target_metric_rows.append({
        "model": MODEL_NAME,
        "seed": SEED,
        "target": target,
        "accuracy": (
            test_metrics[target]["accuracy"]
        ),
        "macro_f1": (
            test_metrics[target]["macro_f1"]
        ),
        "weighted_f1": (
            test_metrics[target]["weighted_f1"]
        ),
        "support": 1050,
    })

target_metrics_df = pd.DataFrame(
    target_metric_rows
)

aggregate_metrics_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "seed": SEED,
    "mean_macro_f1": (
        test_metrics["aggregate"][
            "mean_macro_f1"
        ]
    ),
    "joint_exact_match": (
        test_metrics["aggregate"][
            "joint_exact_match"
        ]
    ),
    "support": 1050,
}])

target_metrics_path = (
    METRICS_DIR
    / "koelectra_seed42_target_metrics.csv"
)

aggregate_metrics_path = (
    METRICS_DIR
    / "koelectra_seed42_aggregate_metrics.csv"
)

target_metrics_df.to_csv(
    target_metrics_path,
    index=False,
)

aggregate_metrics_df.to_csv(
    aggregate_metrics_path,
    index=False,
)

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "baseline_completed",
    "purpose": (
        "second_model_family_generalization"
    ),
    "model_name": MODEL_NAME,
    "model_family": "ELECTRA",
    "seed": SEED,
    "dataset": (
        f"{DATASET_NAME}:{DATASET_CONFIG}"
    ),
    "split_sizes": {
        "train": 8400,
        "validation": 1050,
        "test": 1050,
    },
    "best_epoch": best_epoch,
    "best_valid_mean_macro_f1": (
        best_valid_mean_macro_f1
    ),
    "test_metrics": test_metrics,
    "training_history": training_history,
    "checkpoint_path": str(checkpoint_path),
    "private_prediction_path": str(
        private_prediction_path
    ),
    "privacy": {
        "raw_facts_saved": False,
        "per_case_predictions_private": True,
        "aggregate_metrics_publishable": True,
    },
}

summary_path = (
    RESULT_ROOT / "experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Final output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("KoELECTRA TEST RESULTS")
print("=" * 72)

print(
    target_metrics_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nAGGREGATE RESULTS")
print(
    aggregate_metrics_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nBest validation epoch:", best_epoch)
print(
    "Best validation mean Macro-F1:",
    f"{best_valid_mean_macro_f1:.6f}",
)

print("\nSaved:")
print(" Checkpoint:", checkpoint_path)
print(
    " Private predictions:",
    private_prediction_path,
)
print(" Target metrics:", target_metrics_path)
print(
    " Aggregate metrics:",
    aggregate_metrics_path,
)
print(" Summary:", summary_path)

print("\nPRIVACY")
print("- Do not upload private predictions.")
print("- Do not upload checkpoints for now.")
print("- Aggregate CSV and sanitized JSON may be published later.")

print("\nKoELECTRA BASELINE GENERALIZATION: COMPLETED")
